<a href="https://colab.research.google.com/github/ddoneu/ECON3916-Statistical-Machine-Learning/blob/main/Lab12/%5BLab_12_%5D_OLS%2C_Hedonic_Pricing%2C_and_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/ddoneu/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_ZHVI_2026_Micro.csv'
df = pd.read_csv(url)

df.head()

,Home_Value,Square_Footage,Property_Age,Distance_to_Transit,School_District_Rating
0,329705.74,1941.0,5.5,6.45,Excellent
1,183343.63,1364.3,35.2,2.15,Average
2,354551.73,2386.9,52.4,0.75,Good
3,325773.17,2192.1,50.2,5.25,Excellent
4,359743.12,3069.8,66.5,12.69,Excellent


In [4]:
# Step 2: Defining the formula
# Utilizing the R-style patsy formula interface allows for elegant, readable model specification
formula ='Home_Value ~	Square_Footage	+ Property_Age +	Distance_to_Transit +	School_District_Rating'

In [5]:
# Step 3: Fitting the model and printing the summary
model = smf.ols(formula=formula, data=df)
results = model.fit()
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             Home_Value   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.765
Method:                 Least Squares   F-statistic:                     542.5
Date:                Mon, 16 Mar 2026   Prob (F-statistic):          2.81e-309
Time:                        19:57:26   Log-Likelihood:                -12072.
No. Observations:                1000   AIC:                         2.416e+04
Df Residuals:                     993   BIC:                         2.419e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [6]:
# Step 4: Generating predictions
# We extract the predicted values vector to transition from explanation to prediction
y_pred = results.predict(df)

y_pred


,0
0,312288.586866
1,223813.440910
2,331610.316284
3,307402.426806
4,392714.851722
...,...
995,406437.277604
996,380041.214086
997,236618.275309
998,315508.480776


In [8]:
# Step 5: Calculate RMSE between the actuals and the predictions
model_rmse = rmse(df['Home_Value'], y_pred)

print(f"\nThe Predictive RMSE is: ${model_rmse:,.2f}")


The Predictive RMSE is: $42,316.69


In [9]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go

# ─────────────────────────────────────────────
# SECTION 1: Fit your hedonic OLS model
# ─────────────────────────────────────────────
# Replace this block with your actual DataFrame and feature columns.
# This stub generates synthetic housing data so the dashboard is self-contained.

np.random.seed(42)
n = 300
df = pd.DataFrame({
    "sqft":     np.random.normal(1500, 400, n),
    "bedrooms": np.random.randint(1, 6, n),
    "age":      np.random.randint(0, 50, n),
})
# Introduce mild heteroscedasticity: noise scales with sqft
df["price"] = (
    200_000
    + 120 * df["sqft"]
    + 15_000 * df["bedrooms"]
    - 800 * df["age"]
    + np.random.normal(0, 500 * (df["sqft"] / df["sqft"].mean()), n)  # fan-shaped noise
)

# Add a constant for the intercept term (statsmodels does NOT add one automatically)
X = sm.add_constant(df[["sqft", "bedrooms", "age"]])
y = df["price"]

# Fit the OLS model and store the full results object
model = sm.OLS(y, X)
results = model.fit()

print(results.summary())


# ─────────────────────────────────────────────
# SECTION 2: Extract residuals and fitted values
# ─────────────────────────────────────────────

# results.fittedvalues  → pandas Series of ŷ (predicted values from the training data)
# results.resid         → pandas Series of ε = y - ŷ (raw residuals, same index as y)
fitted     = results.fittedvalues          # ŷ
residuals  = results.resid                 # ε = y - ŷ

# Calculate RMSE from the residuals directly
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"\nRMSE: ${rmse:,.2f}")


# ─────────────────────────────────────────────
# SECTION 3: Outlier detection logic
# ─────────────────────────────────────────────

# Standard deviation of the residuals (ddof=1 for sample std)
resid_std = residuals.std(ddof=1)

# A boolean mask: True where |ε| > 2σ  →  these are our flagged outliers
outlier_mask = residuals.abs() > 2 * resid_std

# Build a clean plot DataFrame with all required columns
plot_df = pd.DataFrame({
    "Fitted Values":  fitted,
    "Residuals":      residuals,
    # Human-readable label for the hover tooltip
    "Point Type":     np.where(outlier_mask, "Outlier (|ε| > 2σ)", "Normal"),
    # Absolute residual magnitude — used to scale marker size for visual weight
    "Abs Residual":   residuals.abs(),
    # Observation index — useful for tracing specific rows back to the raw data
    "Obs Index":      residuals.index,
})


# ─────────────────────────────────────────────
# SECTION 4: Build the interactive Plotly figure
# ─────────────────────────────────────────────

# Color map: outliers get stark crimson, normal points get steel blue
color_map = {
    "Outlier (|ε| > 2σ)": "#DC143C",   # Crimson
    "Normal":              "#4A90D9",   # Steel blue
}

fig = px.scatter(
    plot_df,
    x="Fitted Values",
    y="Residuals",
    color="Point Type",           # Drives the crimson/blue split
    color_discrete_map=color_map,
    size="Abs Residual",          # Larger residual → larger marker (amplifies outlier visibility)
    size_max=18,
    hover_data={
        "Obs Index":     True,
        "Fitted Values": ":,.0f",
        "Residuals":     ":,.0f",
        "Abs Residual":  ":,.0f",
        "Point Type":    True,
    },
    title=(
        f"<b>Residual Forensics Dashboard</b><br>"
        f"<sup>Hedonic Pricing OLS  |  RMSE: ${rmse:,.0f}  |  "
        f"Outliers flagged: {outlier_mask.sum()} of {n} obs ({outlier_mask.mean():.1%})</sup>"
    ),
    labels={
        "Fitted Values": "Fitted Values ŷ  (Predicted Price, $)",
        "Residuals":     "Residuals ε = y − ŷ  ($)",
    },
    template="plotly_white",
)

# ── Zero-reference line ──────────────────────────────────────────────────────
# A perfectly specified model has residuals randomly scattered around ε = 0.
# Any systematic drift above or below this line signals model misspecification.
fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="#555555",
    line_width=1.5,
    annotation_text="ε = 0",
    annotation_position="top right",
    annotation_font_color="#555555",
)

# ── ±2σ reference bands ──────────────────────────────────────────────────────
# Shaded bands make the 2σ threshold explicit and give the eye a boundary to track.
for sign, label_pos in [(1, "top right"), (-1, "bottom right")]:
    fig.add_hline(
        y=sign * 2 * resid_std,
        line_dash="dot",
        line_color="#DC143C",
        line_width=1,
        opacity=0.5,
        annotation_text=f"{'+'if sign>0 else '−'}2σ  (${2*resid_std:,.0f})",
        annotation_position=label_pos,
        annotation_font_color="#DC143C",
        annotation_font_size=11,
    )

# ── Layout polish ────────────────────────────────────────────────────────────
fig.update_layout(
    font_family="Inter, Arial, sans-serif",
    title_font_size=16,
    legend_title_text="Observation Type",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    xaxis=dict(tickprefix="$", tickformat=",.0f", showgrid=True, gridcolor="#EBEBEB"),
    yaxis=dict(tickprefix="$", tickformat=",.0f", showgrid=True, gridcolor="#EBEBEB",
               zeroline=False),   # zeroline=False because we added add_hline manually
    margin=dict(t=100, b=60, l=80, r=40),
    hoverlabel=dict(bgcolor="white", font_size=12),
)

fig.update_traces(marker=dict(opacity=0.75, line=dict(width=0.5, color="white")))

fig.show()

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
Method:                 Least Squares   F-statistic:                 9.988e+05
Date:                Mon, 16 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:04:30   Log-Likelihood:                -2311.7
No. Observations:                 300   AIC:                             4631.
Df Residuals:                     296   BIC:                             4646.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2.001e+05    147.376   1357.581      0.0